# Day 2 — Data Cleaning & Preprocessing (Final)
House Price Prediction (Gujrat, Gujranwala, Sialkot — Zameen.com)

## 1. Data load karein

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw_data_detailed.csv")
print("Shape:", df.shape)
df.head()

## 2. Duplicates hatayein

In [ ]:
before = len(df)
df = df.drop_duplicates(subset="Listing_URL")
print(f"Removed {before - len(df)} duplicates. Remaining: {len(df)}")

## 3. Missing values handle karein
Essential columns: Price, Size, Bedrooms, Bathrooms, Location.

In [ ]:
before = len(df)
df = df.dropna(subset=["Price_PKR", "Size_Marla", "Bedrooms", "Bathrooms", "Location"])
print(f"Dropped {before - len(df)} rows with missing essentials. Remaining: {len(df)}")

## 4. Data types fix karein

In [ ]:
df["Price_PKR"] = pd.to_numeric(df["Price_PKR"], errors="coerce")
df["Size_Marla"] = pd.to_numeric(df["Size_Marla"], errors="coerce")
df["Bedrooms"] = pd.to_numeric(df["Bedrooms"], errors="coerce")
df["Bathrooms"] = pd.to_numeric(df["Bathrooms"], errors="coerce")

before = len(df)
df = df.dropna(subset=["Price_PKR", "Size_Marla", "Bedrooms", "Bathrooms"])
print(f"Dropped {before - len(df)} rows with invalid numeric data. Remaining: {len(df)}")

df["City"] = df["City"].astype("category")

## 5. Outliers remove karein (IQR method)

In [ ]:
def remove_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return data[(data[column] >= lower) & (data[column] <= upper)]

before = len(df)
for col in ["Price_PKR", "Size_Marla", "Bedrooms", "Bathrooms"]:
    df = remove_outliers_iqr(df, col)
print(f"Removed {before - len(df)} outlier rows. Remaining: {len(df)}")

## 6. Feature engineering

In [ ]:
df["Price_per_Marla"] = df["Price_PKR"] / df["Size_Marla"]

df["Area"] = df["Location"].str.split(",").str[0].str.strip()
TOP_N_AREAS = 15
top_areas = df["Area"].value_counts().head(TOP_N_AREAS).index
df["Area_Grouped"] = df["Area"].where(df["Area"].isin(top_areas), "Other").astype("category")

df[["Title", "City", "Bedrooms", "Bathrooms", "Area_Grouped", "Price_PKR"]].head()

## 7. Final sanity check

In [ ]:
print("Final shape:", df.shape)
print("\nMissing values check:")
print(df[["Price_PKR", "Size_Marla", "Bedrooms", "Bathrooms"]].isnull().sum())
df[["Price_PKR", "Size_Marla", "Bedrooms", "Bathrooms"]].describe()

## 8. Save clean data

In [ ]:
df.to_csv("../data/clean_data.csv", index=False)
print(f"Saved {len(df)} clean rows to data/clean_data.csv")
print("Columns:", df.columns.tolist())